# CHIME/FRB Catalog 2 — Non-repeating FRBs
**Nonparametric energy function and cosmic event-rate evolution**

Inputs: `NonrepeatingFRBsdata.txt`, with two columns: estimated redshift $z$ and isotropic-equivalent energy $E$ (erg). The entire calculation is performed in memory. Run cells from top to bottom.

**Method:** Efron–Petrosian $\tau$ statistic; Lynden-Bell $C^{-}$ associated sets; three-parameter `emcee` MCMC power-law fit $\rho(z)=A(1+z)^B$. The original script's definitions of the associated sets, $z_{\max}$ scan, and zero-crossing midpoint are retained. The original script used a grid with $\Delta z=0.01$ to find the **first** crossing of the truncation boundary. This behavior is preserved, including when the de-evolved threshold is not globally monotone.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import cumulative_trapezoid, quad
from scipy.special import logsumexp

plt.rcParams.update({'figure.figsize': (6.2, 4.5), 'font.size': 11})
DATA = Path('NonrepeatingFRBsdata.txt')
if not DATA.exists():
    DATA = Path('/mnt/data/NonrepeatingFRBsdata.txt')  # uploaded-file environment only
z, E = np.loadtxt(DATA, unpack=True)
assert z.ndim == 1 and len(z) > 1 and np.all(z > 0) and np.all(E > 0)
N = len(z)
print(f'Loaded {N} non-repeating FRBs; z range: {z.min():.4g}–{z.max():.4g}')

# Centralized settings. Keep consistent with the published sample selection.
OMEGA_M, H0, C = 0.286, 69600.0, 3e8    # H0 in m/s/Mpc
MPC_CM = 3.08568e24
FLIM = 0.1            # Jy ms: original catalog-2 NR analysis threshold
BANDWIDTH_MHZ = 600
EXPOSURE_YR = 5       # original script's rate normalization
K_GRID = np.linspace(5.9, 6.5, 100)
Z_SCAN = np.arange(0, 10.01, 0.01)

# Equation of the original energy-completeness boundary.
def inv_Ez(x):
    return 1.0 / np.sqrt(OMEGA_M*(1.0 + x)**3 + 1.0 - OMEGA_M)

def luminosity_distance(x):  # cm; supports scalar or array
    x = np.asarray(x)
    return MPC_CM * C / H0 * (1+x) * np.interp(x, distance_grid, radial_grid)

# Pre-tabulate the cosmological integral in RAM (no auxiliary text files).
distance_grid = np.linspace(0, 10, 100001)
radial_grid = cumulative_trapezoid(inv_Ez(distance_grid), distance_grid, initial=0)

def energy_limit(x):
    return 1e39 * 4*np.pi*(luminosity_distance(x)/1e28)**2 * FLIM * (BANDWIDTH_MHZ/1000) / (1+x)

E_lim = energy_limit(z)
E_lim_grid = energy_limit(Z_SCAN)
print(f'Fluence threshold: {FLIM:g} Jy ms; observation span: {EXPOSURE_YR:g} yr')

## 1. Energy–redshift distribution

In [ ]:
fig, ax = plt.subplots()
ax.scatter(z, E, s=7, alpha=.35, label='Non-repeating FRBs')
zg = np.linspace(0, z.max(), 500)
ax.plot(zg, energy_limit(zg), 'k--', lw=1.4, label='Adopted completeness boundary')
ax.set(xlabel='Redshift $z$', ylabel='Isotropic-equivalent energy $E$ (erg)', yscale='log')
ax.legend(frameon=False)
plt.show()

## 2. $k$–$\tau$ scan
For each $k$, compute $E_0=E/(1+z)^k$. The function below preserves the original strict comparisons, unit-offset associated-set counts, and first-crossing $z_{\max}$ grid scan. Pair counting is vectorized; **$k$ is still scanned explicitly**, not estimated using a root optimizer. The first sign-changing pair is averaged to obtain $k_{\mathrm{best}}$.

In [ ]:
def associated_sets(k, return_all=False):
    e0 = E / (1+z)**k
    e0_lim = E_lim / (1+z)**k
    threshold = E_lim_grid / (1+Z_SCAN)**k

    # Equivalent to the original "for q in np.arange(0,10.01,0.01):
    #     if Relation(q)/EvoFc(q,k) > E0: break" for every burst.
    crosses = threshold[None, :] > e0[:, None]
    any_crossing = crosses.any(axis=1)
    first_index = crosses.argmax(axis=1)
    zmax = np.where(any_crossing, Z_SCAN[first_index], Z_SCAN[-1])

    higher_energy = e0[None, :] > e0[:, None]   # [i,j]: E0_j > E0_i
    below_zmax = z[None, :] < zmax[:, None]
    below_zi = z[None, :] < z[:, None]
    Ni = 1 + np.sum(higher_energy & below_zmax, axis=1)
    Mi = 1 + np.sum((e0[None, :] > e0_lim[:, None]) & below_zi, axis=1)
    Ri = 1 + np.sum(higher_energy & below_zi, axis=1)
    if return_all:
        return e0, zmax, Ni, Mi, Ri
    mean = (1+Ni)/2
    variance = (Ni-1)**2/12
    return np.sum(Ri-mean) / np.sqrt(np.sum(variance))

tau_grid = np.array([associated_sets(k) for k in K_GRID])
indices = np.flatnonzero(tau_grid[:-1]*tau_grid[1:] < 0)
if not len(indices):
    raise RuntimeError('No tau=0 sign change within K_GRID. Expand the grid and rerun.')
i0 = indices[0]
k_best = (K_GRID[i0]+K_GRID[i0+1])/2  # original midpoint convention
tau_mid = (tau_grid[i0]+tau_grid[i0+1])/2

def crossing_at(target):
    q = tau_grid-target
    ids = np.flatnonzero(q[:-1]*q[1:] <= 0)
    if not len(ids):
        return np.nan
    i = ids[0]
    return np.interp(target, sorted(tau_grid[i:i+2]),
                     K_GRID[i:i+2][np.argsort(tau_grid[i:i+2])])

k_minus, k_plus = crossing_at(-1), crossing_at(1)
print(f'k = {k_best:.3f} +{k_plus-k_best:.3f} -{k_best-k_minus:.3f}; midpoint tau={tau_mid:+.4f}')
fig, ax = plt.subplots()
ax.plot(K_GRID, tau_grid, lw=1.8)
for val in [-1,0,1]: ax.axhline(val, color='gray', ls='--', lw=.8)
ax.axvline(k_best, color='tab:red', ls=':', label=f'$k={k_best:.3f}$')
ax.set(xlabel='$k$', ylabel=r'$\tau(k)$')
ax.legend(frameon=False)
plt.show()

## 3. Lynden-Bell $C^{-}$ cumulative functions
Use the **same $N_i$ and $M_i$ definitions** as the original code:
$\Psi(E_{0,i})=\prod_{j:E_{0,j}>E_{0,i}}(1+1/N_j)$ and
$\Phi(z_i)=\prod_{j:z_j<z_i}(1+1/M_j)$.
Products are evaluated as sums of logarithms for numerical stability.

In [ ]:
E0, zmax, Ni, Mi, Ri = associated_sets(k_best, return_all=True)
order_E = np.argsort(E0)
order_z = np.argsort(z)
# Each term uses the Ni/Mi belonging to j (not i), as in original notebook.
log_psi = np.array([np.log1p(1/Ni[E0 > ei]).sum() for ei in E0])
log_phi = np.array([np.log1p(1/Mi[z < zi]).sum() for zi in z])
Psi, Phi = np.exp(log_psi), np.exp(log_phi)
sorted_E, sorted_Psi = E0[order_E], Psi[order_E]
sorted_Z, sorted_Phi = z[order_z], Phi[order_z]

fig, axs = plt.subplots(1, 2, figsize=(11.5,4.1))
axs[0].plot(sorted_E, sorted_Psi)
axs[0].set(xscale='log', yscale='log', xlabel='$E_0$ (erg)', ylabel=r'$\Psi(>E_0)$', title='Cumulative energy function')
axs[1].plot(sorted_Z, sorted_Phi)
axs[1].set(xlabel='Redshift $z$', ylabel=r'$\Phi(<z)$', title='Cumulative redshift distribution')
plt.tight_layout(); plt.show()

## 4. Event rate
Reconstruct the original finite-difference estimator using the same bin endpoints (1-based indices). The output is in $\mathrm{yr^{-1}\,Mpc^{-3}}$. The overall normalization depends on exposure, sky coverage and the definition of the $C^{-}$ cumulative function; the power-law slope is insensitive to a common multiplicative normalization. The original notebook used a specific $c/H_0$ numerical approximation in $dV/dz$; we retain that constant below for continuity.

In [ ]:
# Original 1-based endpoints; they are data/sample-specific.
BIN_ENDPOINTS = np.array([1, 200, 425, 675, 850, 1000, 1260, 1450,
                          1700, 2000, 2200, 2400, 2700, 2800, 2900,
                          2950, 3000, 3020, 3045, N])
assert np.all(np.diff(BIN_ENDPOINTS)>0) and BIN_ENDPOINTS[-1] <= N
idx = BIN_ENDPOINTS-1
z_left, z_right = sorted_Z[idx[:-1]], sorted_Z[idx[1:]]
dphi = sorted_Phi[idx[1:]]-sorted_Phi[idx[:-1]]
meanz = (z_left+z_right)/2
slope = dphi/(z_right-z_left)
# dV/dz in cm^3, matches original notebook's 1.32243428571429E+28 factor.
dVdz = 1.32243428571429e28 * 4*np.pi * (luminosity_distance(meanz)/(1+meanz))**2 * inv_Ez(meanz)
rho = MPC_CM**3 * slope * (1+meanz) / (dVdz*EXPOSURE_YR)
rho_err = rho/np.sqrt(dphi)
valid = np.isfinite(rho) & np.isfinite(rho_err) & (rho>0) & (rho_err>0)
meanz, rho, rho_err = meanz[valid], rho[valid], rho_err[valid]
print(f'Rate bins: {len(rho)}; z range: {meanz.min():.3f}–{meanz.max():.3f}')

fig, ax = plt.subplots()
ax.errorbar(1+meanz, rho, yerr=rho_err, fmt='o', ms=4, capsize=2)
ax.set(xscale='log', yscale='log', xlabel='$1+z$', ylabel=r'$\rho(z)$ (yr$^{-1}$ Mpc$^{-3}$)', title='Nonparametric event-rate estimate')
plt.show()

## 5. Event-rate power-law fit (original three-parameter MCMC)

Fit the **non-repeating** sample only, using the same model as the manuscript fitting script: $\log_{10}\rho=\log_{10}A+B\log_{10}(1+z)$ with additional intrinsic scatter $\exp(\ln f)$ in log-rate space. The data and error bars are both normalized by $\max(\rho)$ before taking logarithms; the slope $B$ is unaffected by this normalization. The Gaussian log-likelihood, uniform prior, initial guess/bounds, Nelder–Mead initialization and `emcee` settings match the supplied three-sample code. The nonparametric event-rate values are fed directly from the previous cell (no intermediate `*-rou.txt`).

Dependencies: `pip install emcee corner`. Corner and rate-fit plots are displayed **inline**, not saved automatically.

In [ ]:
import emcee
import corner
from scipy.optimize import minimize

# Normalize as in the original fitting code (single NR sample).
sort_idx = np.argsort(meanz)
x_rate = 1.0 + meanz[sort_idx]
y_rate = rho[sort_idx].copy()
y_rate_err = rho_err[sort_idx].copy()
norm = y_rate.max()
y_rate, y_rate_err = y_rate / norm, y_rate_err / norm

logx = np.log10(x_rate)
logy = np.log10(y_rate)
logy_err = y_rate_err / (y_rate * np.log(10))

# Same NR guess and uniform bounds: (log10 A, B, ln f).
guess = np.array([-6.8, -5.5, -2.3])
bounds = np.array([-15., 0., -15., 5., -10., 1.])

def log_likelihood(theta, lx, ly, ly_err):
    logA, gamma, ln_f = theta
    model = logA + gamma * lx
    sigma2 = ly_err**2 + np.exp(2 * ln_f)
    if np.any(sigma2 <= 0) or not np.all(np.isfinite(model)):
        return -np.inf
    return -0.5 * np.sum((ly - model)**2 / sigma2 + np.log(2*np.pi*sigma2))

def log_prior(theta):
    logA, gamma, ln_f = theta
    a_min, a_max, g_min, g_max, f_min, f_max = bounds
    return 0. if (a_min < logA < a_max and g_min < gamma < g_max and f_min < ln_f < f_max) else -np.inf

def log_probability(theta, lx, ly, ly_err):
    lp = log_prior(theta)
    return lp + log_likelihood(theta, lx, ly, ly_err) if np.isfinite(lp) else -np.inf

# Initialize with the same Nelder–Mead strategy, then sample with emcee.
args = (logx, logy, logy_err)
soln = minimize(lambda theta: -log_probability(theta, *args),
                guess, method='Nelder-Mead')
center = soln.x if soln.success and np.isfinite(log_prior(soln.x)) else guess
ndim, nwalkers = 3, 32
rng = np.random.default_rng(2026)  # reproducible initialization
pos = center + 1e-2 * rng.standard_normal((nwalkers, ndim))
for i in range(nwalkers):
    if not np.isfinite(log_prior(pos[i])):
        pos[i] = guess + 1e-3 * rng.standard_normal(ndim)

sampler = emcee.EnsembleSampler(nwalkers, ndim, log_probability, args=args)
sampler.run_mcmc(pos, 5000, progress=True)
flat_samples = sampler.get_chain(discard=1000, thin=15, flat=True)
quantiles = np.percentile(flat_samples, [16, 50, 84], axis=0)
median = quantiles[1]
err_down, err_up = median - quantiles[0], quantiles[2] - median

for label, j in [('log10 A', 0), ('B', 1), ('ln f', 2)]:
    print(f'{label:7s} = {median[j]:.4f} +{err_up[j]:.4f} -{err_down[j]:.4f}')
print(f'Mean acceptance fraction: {np.mean(sampler.acceptance_fraction):.3f}')

fig = corner.corner(flat_samples,
                    labels=[r'$\log_{10}A$', r'$B$', r'$\ln f$'],
                    truths=median, show_titles=True)
plt.show()

# Paper-style normalized rate and posterior fit for NR only.
fig, ax = plt.subplots(figsize=(7, 5))
ax.step(x_rate, y_rate, where='mid', color='#3bb293', lw=2.5,
        label=r'$S_{\mathrm{NR}}$')
ax.errorbar(x_rate, y_rate, yerr=y_rate_err, fmt='none',
            color='#3bb293', lw=1.5)
xfit = np.geomspace(x_rate.min(), x_rate.max(), 300)
logxfit = np.log10(xfit)
ax.plot(xfit, 10**(median[0] + median[1]*logxfit),
        ls='--', color='#167f66', lw=2,
        label=rf'$B={median[1]:.2f}^{{+{err_up[1]:.2f}}}_{{-{err_down[1]:.2f}}}$')
for pars in flat_samples[rng.choice(len(flat_samples), size=100, replace=False)]:
    ax.plot(xfit, 10**(pars[0] + pars[1]*logxfit),
            color='#3bb293', alpha=.025, lw=1)
ax.set(xscale='log', yscale='log', xlabel='$1+z$',
       ylabel=r'Normalized $\rho(z)$')
ax.legend(frameon=False)
plt.show()

### Reproducibility notes
- Input is the 3057-object NR sample; the nonparametric rate points `meanz`, `rho`, `rho_err` are passed in memory to MCMC.
- The original rate reconstruction, $k$–$\tau$ scan and bin endpoints are unchanged.
- Unlike the previous two-parameter demonstration, this fit includes $\ln f$ and uses the manuscript script's NR priors and `emcee` configuration (`32` walkers, `5000` steps, `1000` discarded, thinning `15`).
- The `corner` plot and fit plot are shown inline; no auxiliary files are written. A common scaling of the event rates leaves $B$ unchanged, but the precise posterior depends on the actual binning and uncertainties.
- Install: `pip install numpy scipy matplotlib emcee corner`.